# Полный поиск размещения школы в Тюмени

**Запускать вручную:** выберите Python `.venv-blocksnet` и выполните кодовые ячейки **1 → 2 → 3 → 4**. Ячейки 1–2 только подключают сервер и проверяют условия. **Ячейка 3 запускает или продолжает полный поиск**, ячейка 4 показывает ход расчёта под собой. Ячейка 5 — результаты, ячейка 6 — необязательная остановка.

Одна школа **600 мест**, независимые варианты во всех **12 805 кварталах**. Работают исходный движок приложения и его CBC: BlocksNet 1.0.0a9, 120 мест на 1000 жителей, 15 минут пешком, `self_supply=True`, `max_depth=1`. Население, школы, нормативы, А/Б и метод не меняются.

Результаты записывает сервер в `state/application.sqlite`, как при запуске из браузера. После перезагрузки снова выполните 1–4: найдётся прежнее задание, уже сохранённые кварталы не рассчитываются повторно. Если сервер уже доступен, второй процесс не создаётся; иначе ячейка 2 запускает его в фоне. Сервер продолжает работу после закрытия ноутбука, пока компьютер включён. Не переводите компьютер в сон во время расчёта.

**Подготовка проверена без запуска рабочего полного поиска.** Прогноз по прежнему ограниченному замеру — около 77–96 минут; фактическая оценка обновляется во время поиска. Оценки населения и вместимости остаются модельными, разной достоверности.

## 1. Python и подключение клиента

В VS Code: **Select Kernel → Python Environments →**
`D:\Рабочий стол\ITMO AI\python-package-boilerplate\.venv-blocksnet\Scripts\python.exe`.

Никакие библиотеки заново устанавливать не требуется. Здесь поиск не запускается. `SEARCH_JOB_ID = None` означает выбор последнего совместимого поиска; задавать ID вручную обычно не нужно.

In [ ]:
from pathlib import Path
import importlib.util
import sys

PROJECT = Path(r"D:\Рабочий стол\ITMO AI\school-placement")
PYTHON = Path(r"D:\Рабочий стол\ITMO AI\python-package-boilerplate\.venv-blocksnet\Scripts\python.exe")
PORT = 8767
SEARCH_JOB_ID = None

if str(Path(sys.executable).resolve()).casefold() != str(PYTHON.resolve()).casefold():
    raise RuntimeError(f"Выберите ядро {PYTHON}. Сейчас используется: {sys.executable}")
spec = importlib.util.spec_from_file_location("school_notebook_search", PROJECT / "notebook_search.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
client = module.NotebookSearch(PROJECT, port=PORT)
print("Python:", sys.executable)
print("Приложение:", client.base_url)
print("Клиент загружен. Поиск не запущен этой ячейкой.")

## 2. Сервер и обязательные проверки

Проверяются сохранённые А/Б и обоснования, база результатов, подпись исходных данных, отчёт эквивалентности и отсутствие другого активного расчёта. А — № 9345, 600 мест; Б — № 9736 и № 477, по 300 мест. Если уже выполняется тот же совместимый поиск, он будет использован без нового задания.

При холодном запуске дождитесь загрузки матрицы. Журнал: `state/notebook-server.log`. Если база уже открыта сервером на другом порту, укажите его в `PORT` и повторите 1–2. Не запускайте второй сервер вручную.

In [ ]:
client.ensure_server()
checks = client.preflight()

## 3. Запустить или продолжить полный поиск

**Эта ячейка запускает расчёт.** Повторное выполнение использует существующее задание: продолжает прерванное, подключается к работающему или открывает завершённое. Все варианты начинаются от исходного состояния; вместимость между ними не накапливается. Результаты каждого квартала сохраняет существующий серверный обработчик.

Ошибка отдельного варианта сохраняется в рейтинге и не маскируется. Если обход завершится с ошибками, ноутбук не объявит его полным успешным перебором и не будет автоматически повторять уже записанные варианты.

In [ ]:
SEARCH_JOB_ID = client.start_or_resume(SEARCH_JOB_ID)
print("ID задания:", SEARCH_JOB_ID)
print("Теперь выполните ячейку 4 для отображения прогресса.")

## 4. Прогресс под ячейкой

Показатели обновляются каждые 3 секунды: число кварталов, процент, прошедшее время расчётов, оценка оставшегося времени и ошибки. Прошедшее время — сумма времени сохранённых вариантов: перерывы между запусками в него не входят. Оценка остатка основана на уже рассчитанных вариантах и приблизительна.

**Interrupt / остановка ячейки прерывает только наблюдение, сервер продолжает считать.** Повторите эту ячейку для наблюдения или используйте ячейку 6 для остановки самого поиска. После потери связи/перезагрузки повторите 1–4. Полным считается только успешный расчёт всех кварталов; лучший среди части вариантов не выдаётся за глобальный.

In [ ]:
if SEARCH_JOB_ID is None:
    raise RuntimeError("Сначала выполните ячейки 1–3.")
final_state = client.watch(SEARCH_JOB_ID, interval=3)

## 5. Просмотр результата

Не запускает новых расчётов. Полный рейтинг, все равноценные лучшие и варианты на карте доступны в том же браузерном приложении. Таблица ниже ограничена 20 строками; ссылка на CSV содержит весь сохранённый рейтинг.

In [ ]:
from IPython.display import display, HTML
import pandas as pd

if SEARCH_JOB_ID is None:
    raise RuntimeError("Сначала выполните ячейки 1–3.")
result = client.job(SEARCH_JOB_ID)
display(HTML(client.progress_html(result)))
if result.get("best"):
    print("Все равноценные лучшие:" if result.get("exhaustive") else "Лучшие только среди успешно проверенных:", len(result["best"]))
    display(pd.DataFrame(result["best"]).head(20).rename(columns={
        "id": "Квартал", "deficit": "Дефицит, мест", "reduction": "Сокращение, мест",
        "error": "Ошибка", "seconds": "Время, с",
    }))
print("Открыть приложение:", client.base_url)
print("Рейтинг CSV:", f"{client.base_url}/api/job?id={SEARCH_JOB_ID}&csv=1")

## 6. Необязательно: остановить поиск с сохранением результатов

Сначала прервите ячейку наблюдения, если она занята. **Измените `STOP_SEARCH` на `True`** и выполните эту ячейку. Сервер остановится после текущего квартала. Для продолжения верните `False` и выполните 3–4 (после перезапуска ядра — 1–4). Эта ячейка по умолчанию ничего не останавливает, в том числе при Run All.

In [ ]:
STOP_SEARCH = False

if STOP_SEARCH:
    if SEARCH_JOB_ID is None:
        raise RuntimeError("Нет выбранного задания. Сначала выполните ячейки 1–3.")
    client.stop(SEARCH_JOB_ID)
else:
    print("Остановка не запрошена. Для остановки измените STOP_SEARCH на True.")